In [23]:
from pathlib import Path
import json
from collections import defaultdict
import itertools

In [24]:
dataset = Path.cwd() / "input" / "test_revised.json"
output_path = Path.cwd() / "output"
output_path.mkdir(exist_ok=True, parents=True)

print(dataset)

with open(dataset, "r", encoding="utf-8") as f:
    jsons = json.load(f)

print(len(jsons))

/home/elpocher/projects/gh/MS-KeBAB/experiments/re_docred_dataset/test_revised.json
500


In [25]:
properties_to_drop = {"pos", "global_pos", "index", "sent_id", "properties"}

def merge_entities(entities: list[dict]):
    merged_entities = {}
    for entity in entities:
        for k, v in entity.items():
            if k not in properties_to_drop:
                if k not in merged_entities:
                    merged_entities[k] = set()
                merged_entities[k].add(v)

    for k, v in merged_entities.items():
        if len(v) == 1:
            merged_entities[k] = v.pop()
        else:
            merged_entities[k] = list(v)

    return merged_entities

In [27]:
output_dataset = []
for entry in jsons:
    # vertexSet: list[list[dict]]. A list of entities, where each element of the list is another list of mentions of the same entity. 
    # labels: ?
    # sents: list of sentences represented as a list[list[str]]. Each sentence is a list of words.
    text = " ".join([" ".join(sent[:-1]) + sent[-1] for sent in entry["sents"]])
    target_entities = [merge_entities(entry) for entry in entry["vertexSet"]]
    print(target_entities)
    print(len(target_entities)) # 15
    print(len(entry["labels"])) # 11
    print(text) # 7
    # print([len(sent) for sent in entry["sents"]])
    # print(min([e["t"] for e in entry["labels"]]), max([e["t"] for e in entry["labels"]])) # 0 14
    # print(min([e["h"] for e in entry["labels"]]), max([e["h"] for e in entry["labels"]])) # 0 10
    properties = []
    for p in entry["labels"]:
        entity_index, index, property_id = p["t"], p["h"], p["r"]
        target_entities[index][property_id] = target_entities[entity_index]["name"]

    output_entry = {
        "id": hash(text),
        "title": entry["title"],
        "text": text,
        "target_entities": target_entities,
    }

    output_dataset.append(output_entry)


[{'name': ['Loud Tour', 'Loud'], 'type': 'MISC'}, {'name': 'Barbadian', 'type': 'LOC'}, {'name': 'Rihanna', 'type': 'PER'}, {'name': 'twenty', 'type': 'NUM'}, {'name': 'Americas', 'type': 'LOC'}, {'name': 'Europe', 'type': 'LOC'}, {'type': 'TIME', 'name': '2010'}, {'name': 'United Kingdom', 'type': 'LOC'}, {'name': 'London', 'type': 'LOC'}, {'type': 'NUM', 'name': '10'}, {'type': 'LOC', 'name': 'The O2 Arena'}, {'type': 'NUM', 'name': 'US$90 million'}, {'type': 'NUM', 'name': '98'}, {'type': 'NUM', 'name': '1,200,800'}, {'type': 'TIME', 'name': '2011'}]
15
11
The Loud Tour was the fourth overall and third world concert tour by Barbadian recording artist Rihanna. Performing in over twenty countries in the Americas and Europe , the tour was launched in support of Rihanna 's fifth studio album Loud ( 2010 ). Critics acclaimed the show for its liveliness and higher caliber of quality when compared to Rihanna 's previous tours. The Loud Tour was a large commercial success , experiencing dem

In [28]:
output_file = output_path / "test.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for entry in output_dataset:
        print(entry)
        try:
            json.dump(entry, f)
            f.write('\n')
        except:
            print("Failed to save an entry")

{'id': 4217417708826597183, 'title': 'Loud Tour', 'text': "The Loud Tour was the fourth overall and third world concert tour by Barbadian recording artist Rihanna. Performing in over twenty countries in the Americas and Europe , the tour was launched in support of Rihanna 's fifth studio album Loud ( 2010 ). Critics acclaimed the show for its liveliness and higher caliber of quality when compared to Rihanna 's previous tours. The Loud Tour was a large commercial success , experiencing demand for an extension of shows in the United Kingdom due to popularity. In London , Rihanna played a record breaking 10 dates at The O2 Arena. The tour ultimately grossed an estimated value of US$ 90 million from 98 reported shows and a total audience of 1,200,800. The Loud Tour became the seventh - highest grossing tour of 2011.", 'target_entities': [{'name': ['Loud Tour', 'Loud'], 'type': 'MISC', 'P577': '2011', 'P175': 'Rihanna'}, {'name': 'Barbadian', 'type': 'LOC'}, {'name': 'Rihanna', 'type': 'PER